- 머신 러닝 기본적인 절차
  - CSV 읽기 -> 입력(X)과 정답(y) 분리 -> LDA 학습(fit) -> 예측(predict)과 확률(predict_proba)
  - 조금 더 세부적으로는 데이터를 읽고 결측치, 이상치 등을 체크하고 값을 변환하거나 스케일링 등의 과정을 거치는 경우도 있음

In [1]:
"""
분류
    와인 품종(3종): 선형 판별 분석(LDA)
    - 이탈리아 같은 지역 3개 농장의 와인 178개, 화학성분 13개
    - LDA: 클래스끼리 가장 잘 떨어지는 방향(축)을 찾아 그 축 위에서 분류
    - 흐름: 데이터 읽기 -> 모델 학습(fit) -> 새 데이터 예측(predict)
"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/wine.csv"
randomSeed = 42

# 
cols = ["alcohol", "malic_acid", "ash", "alcalinity", "magesium", "phenols",
       "flavanoids", "nonflav_phenols", "proanthocyanins", "color_intersity",
       "hue", "od280_df315", "proline", "cultivar"]

wine = pd.read_csv(url, header=None, names=cols)

X = wine.drop("cultivar", axis=1)
y = wine["cultivar"]            # 1, 2, 3

wine.head()

,alcohol,malic_acid,ash,alcalinity,magesium,phenols,flavanoids,nonflav_phenols,proanthocyanins,color_intersity,hue,od280_df315,proline,cultivar
0,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065,1
1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050,1
2,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185,1
3,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480,1
4,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735,1


In [2]:
print("전체 데이터", X.shape, "| 품종별 개수:", y.value_counts().sort_index().to_dict())

전체 데이터 (178, 13) | 품종별 개수: {1: 59, 2: 71, 3: 48}


In [4]:
# 학습/테스트 데이터 분할
# stratify=y: 품종 비율을 train/test 에 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=randomSeed
)

print("train: ", X_train.shape, "| test: ", X_test.shape)

train:  (124, 13) | test:  (54, 13)


In [6]:
# 학습 (train 데이터만 사용)
model = LinearDiscriminantAnalysis()
model.fit(X_train, y_train)

LinearDiscriminantAnalysis()

In [11]:
# 평가
pred = model.predict(X_test)
print("\ntrain 정확도: ", round(model.score(X_train, y_train), 3))
print("test 정확도: ", round(accuracy_score(y_test, pred), 3))
print("혼동 행렬 (행: 실제, 열: 예측):\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))


train 정확도:  1.0
test 정확도:  0.981
혼동 행렬 (행: 실제, 열: 예측):
 [[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]
              precision    recall  f1-score   support

           1      0.947     1.000     0.973        18
           2      1.000     0.952     0.976        21
           3      1.000     1.000     1.000        15

    accuracy                          0.981        54
   macro avg      0.982     0.984     0.983        54
weighted avg      0.982     0.981     0.982        54



In [14]:
# 예측
sample = X_test.sample(3, random_state=randomSeed)
print("실제 품종: ", y_test.loc[sample.index].tolist())
print("예측 품종: ", model.predict(sample).tolist())
print("품종별 확률:\n", model.predict_proba(sample).round(3))

실제 품종:  [3, 1, 3]
예측 품종:  [3, 1, 3]
품종별 확률:
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]]
